# Birdseye Camera Setup

Run through this notebook once to fully configure a new Birdseye Camera. All calibration files are saved to `peripherals/configs/` so that `BirdseyeCamera.from_config()` loads everything automatically in future sessions.

**Sections:**
1. Configuration — create your config file
2. Lens calibration — intrinsic camera parameters
3. Generate ChArUco boards — for machine calibration
4. Machine calibration — map pixels to machine coordinates
5. Verify — confirm accuracy
6. Future sessions — one-line setup

---
## 0. Imports and settings

Edit the values in the cell below to match your setup, then run it.

In [ ]:
import glob
import json
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

from science_jubilee.peripherals.BirdseyeCamera import BirdseyeCamera

# ── Edit these ────────────────────────────────────────────────────────────────
CONFIG_NAME     = "MyCamera_config.json"   # filename for your config
CAMERA_INDEX    = 0                        # OpenCV device index
RESOLUTION      = (3264, 2448)            # (width, height) pixels
MACHINE_ADDRESS = "192.168.1.2"            # Duet IP address
TOOL_INDEX      = 0                        # tool index to use for jogging
# ─────────────────────────────────────────────────────────────────────────────

# Derived paths — no need to edit
_bc_file    = sys.modules['science_jubilee.peripherals.BirdseyeCamera'].__file__
CONFIGS_DIR = os.path.normpath(os.path.join(os.path.dirname(_bc_file), 'configs', 'user'))
LENS_IMAGE_DIR   = os.path.join(os.getcwd(), 'lens_cal_images')
LENS_CAL_FILE    = 'lens_calibration.npz'
MACHINE_CAL_FILE = 'camera_calibration.npz'

print(f"Configs dir (user): {CONFIGS_DIR}")

---
## 1. Create config

Creates `peripherals/configs/user/<CONFIG_NAME>`. Re-running will overwrite.

In [ ]:
BirdseyeCamera.create_config(
    CONFIG_NAME,
    camera_index=CAMERA_INDEX,
    resolution=RESOLUTION,
)
print("Calibration file paths will be added to the config automatically after Steps 2 and 4.")

---
## 2. Lens calibration

Corrects for lens distortion and measures focal length. You need a **printed checkerboard** (default: 9×6 board, 25 mm squares).

**Collect 20–30 images:**
- Cover all four corners of the frame and the centre
- Tilt the board ~30–45° in both X and Y a few times
- Vary the distance slightly

Run **preview** to see the camera view, reposition the board, then run **capture**. Repeat until you have 20+ images.

In [ ]:
# Preview — shows current camera view
os.makedirs(LENS_IMAGE_DIR, exist_ok=True)
n_saved = len(glob.glob(os.path.join(LENS_IMAGE_DIR, '*.png')))

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, RESOLUTION[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, RESOLUTION[1])
for _ in range(5): cap.read()
ret, frame = cap.read()
cap.release()

if ret:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(f"{n_saved} images saved — position checkerboard, then run 'capture' cell below")
    plt.tight_layout()
    plt.show()
else:
    print(f"Could not open camera at index {CAMERA_INDEX}")

In [ ]:
# Capture — run this cell each time you want to save the current frame
n_saved = len(glob.glob(os.path.join(LENS_IMAGE_DIR, '*.png')))

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, RESOLUTION[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, RESOLUTION[1])
for _ in range(5): cap.read()
ret, frame = cap.read()
cap.release()

if ret:
    path = os.path.join(LENS_IMAGE_DIR, f'frame_{n_saved:03d}.png')
    cv2.imwrite(path, frame)
    print(f"Saved {path}  ({n_saved + 1} images total)")
else:
    print("Could not capture frame")

In [ ]:
# Compute lens calibration from saved images and save to configs/
CHECKERBOARD_CORNERS = (8, 5)  # interior corners (cols-1, rows-1) — update to match your board
SQUARE_MM = 25.0               # physical square size in mm — update to match your board

images = sorted(glob.glob(os.path.join(LENS_IMAGE_DIR, '*.png')))
print(f"Found {len(images)} images")
assert len(images) >= 10, "Need at least 10 images — collect more and re-run"

objp = np.zeros((CHECKERBOARD_CORNERS[0] * CHECKERBOARD_CORNERS[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD_CORNERS[0], 0:CHECKERBOARD_CORNERS[1]].T.reshape(-1, 2)
objp *= SQUARE_MM

obj_pts, img_pts, failed = [], [], []
image_size = None
for path in images:
    img  = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    image_size = gray.shape[::-1]
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD_CORNERS, None)
    if ret:
        refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1),
            (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))
        obj_pts.append(objp)
        img_pts.append(refined)
    else:
        failed.append(os.path.basename(path))

print(f"Corners detected: {len(obj_pts)}/{len(images)} images")
if failed: print(f"Detection failed: {', '.join(failed)}")
assert len(obj_pts) >= 10, "Not enough successful detections — collect more images"

rms, camera_matrix, dist_coeffs, _, _ = cv2.calibrateCamera(
    obj_pts, img_pts, image_size, None, None)

quality = "excellent" if rms < 0.5 else "good" if rms < 1.0 else "marginal — consider recollecting" if rms < 2.0 else "poor — recollect images"
print(f"RMS reprojection error: {rms:.4f} px  ({quality})")

out_path = os.path.join(CONFIGS_DIR, LENS_CAL_FILE)
np.savez(out_path, camera_matrix=camera_matrix, dist_coeffs=dist_coeffs)
print(f"Saved to {out_path}")

config_path = os.path.join(CONFIGS_DIR, CONFIG_NAME)
with open(config_path) as f: cfg = json.load(f)
cfg['lens_calibration_path'] = LENS_CAL_FILE
with open(config_path, 'w') as f: json.dump(cfg, f, indent=4)
print("Config updated with lens_calibration_path")

---
## 3. Generate ChArUco boards

Print at **100% scale / 300 DPI** (no fit-to-page). Glue or tape flat to a rigid backing. After printing, **measure the actual square size in mm** — this is what you enter in Step 4.

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath('BirdseyeSetup.ipynb')))
from generate_charuco_board import generate_board

BOARD_COLS      = 9     # squares across
BOARD_ROWS      = 6     # squares down
BOARD_SQUARE_MM = 20.0  # design size in mm (measure actual after printing!)
N_BOARDS        = 1     # number of boards to generate

for i in range(1, N_BOARDS + 1):
    out = os.path.join(os.getcwd(), f'charuco_board_{i}.png')
    generate_board(BOARD_COLS, BOARD_ROWS, BOARD_SQUARE_MM, i, out)

print(f"\nDot A = interior corner (col=0, row=0) — jog here first")
print(f"Dot B = interior corner (col={BOARD_COLS-2}, row=0) — jog here second")

---
## 4. Machine calibration

Maps pixels to machine coordinates across a range of bed Z heights.

**Before running:**
1. Tape the ChArUco board flat on the bed
2. Pick up a tool with a pointed tip for jogging (or just use the z stop)
3. Park or move the tool out of frame before the calibration loop runs

In [ ]:
from science_jubilee.Machine import Machine

machine = Machine(address=MACHINE_ADDRESS)
cam = BirdseyeCamera.from_config(CONFIG_NAME)
cam.attach(machine)

In [ ]:
# Optional: home the machine if needed
# machine.home_all()

In [ ]:
# Optional - Load and pick up your jogging tool — edit to match your setup
# A syringe is useful here, but can also align to the z-stop toolchanger
# from science_jubilee.tools.Syringe import Syringe

# tool = Syringe(index=TOOL_INDEX, name="syringe", config="10cc_syringe_liquidhandling")
# machine.load_tool(tool)
# machine.pickup_tool(tool)

In [ ]:
# Confirm the board is visible
frame = cam.get_frame()
corners, ids, _ = cam.detect_aruco(frame)
print("Detected marker IDs:", ids.flatten() if ids is not None else "none — check board is in frame")
cam.show_frame(cam.draw_detected_markers(frame, corners, ids))

In [ ]:
# Jog to dot A (left dot, one square in from A side), then run this cell
pos = machine.get_position()
corner_a = (float(pos['X']), float(pos['Y']), float(pos['Z']))
print(f"corner_a = {corner_a}")

In [ ]:
# Jog to dot B (right dot, one square in from B side), then run this cell
pos = machine.get_position()
corner_b = (float(pos['X']), float(pos['Y']), float(pos['Z']))
print(f"corner_b = {corner_b}")

In [ ]:
# Park/move tool out of frame, then run the Z-stack calibration
# machine.park_tool()  # uncomment if applicable

SQUARE_SIZE_MM   = 27.0  # ← update to your measured square size after printing
MACHINE_CAL_PATH = os.path.join(CONFIGS_DIR, MACHINE_CAL_FILE)

def calibrate_z_stack(
    cam, machine, corner_a, corner_b,
    z_end=200, dz=5,
    board_cols=9, board_rows=6, square_mm=27,
    save_path=None,
    y_sign=-1,
    pixel_axes=([0, -1, 0], [-1, 0, 0]),
):
    """Calibrate at successive Z heights and build a Z-stack for accurate
    pixel_to_machine at any bed height."""
    a_xy = corner_a[:2]
    b_xy = corner_b[:2]
    z_start = float(machine.get_position()['Z'])
    z_values = np.arange(z_start, z_end + dz, dz)

    cam.connect()
    try:
        succeeded, failed = [], []
        for z in z_values:
            machine.move_to(z=z)
            print(f"\n--- Z={z:.1f} ---")
            try:
                frame = cam.get_frame()
                cam.calibrate_3d_charuco(
                    boards=[((*a_xy, z), (*b_xy, z))],
                    board_cols=board_cols, board_rows=board_rows,
                    square_mm=square_mm, frame=frame,
                    save_path=save_path, y_sign=y_sign, pixel_axes=pixel_axes,
                )
                succeeded.append(z)
            except Exception as e:
                print(f"  SKIPPED: {e}")
                failed.append(z)
    finally:
        cam.disconnect()

    print(f"\nDone: {len(succeeded)} succeeded, {len(failed)} failed")
    if failed:
        print(f"Failed Z: {[round(z, 1) for z in failed]}")

calibrate_z_stack(
    cam, machine, corner_a, corner_b,
    square_mm=SQUARE_SIZE_MM,
    save_path=MACHINE_CAL_PATH,
)

In [ ]:
# Save machine calibration path to config
config_path = os.path.join(CONFIGS_DIR, CONFIG_NAME)
with open(config_path) as f: cfg = json.load(f)
cfg['machine_calibration_path'] = MACHINE_CAL_FILE
with open(config_path, 'w') as f: json.dump(cfg, f, indent=4)
print("Config updated with machine_calibration_path")
print("\nFinal config:")
print(json.dumps(cfg, indent=4))

---
## 5. Verify

Overlays a machine coordinate grid on the camera image. Cyan lines should align with the physical bed.

If the grid is mirrored: re-run `calibrate_z_stack` with `y_sign=+1`.

In [ ]:
%matplotlib widget
import matplotlib.patheffects as pe

Z_CHECK = corner_a[2]
X_RANGE, Y_RANGE, SPACING = (0, 300), (0, 220), 25

frame = cam.get_frame()
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(frame)
ax.axis('off')

outline = [pe.Stroke(linewidth=2, foreground='black'), pe.Normal()]
xs = np.arange(X_RANGE[0], X_RANGE[1] + 1, SPACING)
ys = np.arange(Y_RANGE[0], Y_RANGE[1] + 1, SPACING)

for x in xs:
    pts = [cam.machine_to_pixel(x, y, Z_CHECK) for y in ys]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color='cyan', lw=0.8, alpha=0.7)
for y in ys:
    pts = [cam.machine_to_pixel(x, y, Z_CHECK) for x in xs]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color='cyan', lw=0.8, alpha=0.7)
for x in xs[::2]:
    for y in ys[::2]:
        px, py = cam.machine_to_pixel(x, y, Z_CHECK)
        ax.text(px, py, f"{x:.0f},{y:.0f}", fontsize=5, color='yellow',
                ha='center', va='center', path_effects=outline)

plt.tight_layout()
plt.show()

---
## 6. Future sessions

Lens and machine calibration are loaded automatically from the config:

In [ ]:
# from science_jubilee.peripherals.BirdseyeCamera import BirdseyeCamera
# from science_jubilee.Machine import Machine
#
# machine = Machine(address="192.168.1.2")
# cam = BirdseyeCamera.from_config("MyCamera_config.json")
# cam.attach(machine)